# EchoNet

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import pickle
import os

# --- 1. CONFIGURATION ---
csv_path = "/home/sagemaker-user/user-default-efs/vjepa2/data/echonet/echonetdynamic-2/EchoNet-Dynamic/FileList.csv"
video_root = "/home/sagemaker-user/user-default-efs/vjepa2/data/echonet/echonetdynamic-2/EchoNet-Dynamic/Videos/"
scaler_filename = "ef_scaler.pkl"

# --- 2. LOAD & PREPARE DATA ---
print("Loading CSV...")
df = pd.read_csv(csv_path)

# Construct the full file path for every video
# We use os.path.join for safety, though string formatting works too
df["video_path"] = df["FileName"].apply(lambda x: os.path.join(video_root, f"{x}.avi"))

# Verify files exist (optional sanity check - checks first 5)
for path in df["video_path"].head().values:
    if not os.path.exists(path):
        print(f"WARNING: File not found: {path}")

# --- 3. SPLIT THE DATA ---
# We strictly use the 'Split' column provided by EchoNet
train_df = df[df["Split"] == "TRAIN"].copy()
val_df   = df[df["Split"] == "VAL"].copy()
test_df  = df[df["Split"] == "TEST"].copy()

print(f"Data Split: Train ({len(train_df)}), Val ({len(val_df)}), Test ({len(test_df)})")

# --- 4. NORMALIZE TARGET (EF) ---
scaler = StandardScaler()

# A. FIT ONLY on Training Data
# Reshape is required because scaler expects 2D array [[v1], [v2]...]
train_ef_values = train_df["EF"].values.reshape(-1, 1)
scaler.fit(train_ef_values)

print(f"\nScaler Fitted on Train Set.")
print(f"Mean EF: {scaler.mean_[0]:.4f}")
print(f"Std  EF: {np.sqrt(scaler.var_[0]):.4f}")

# B. TRANSFORM All Sets
# We create a new column 'EF_normalized' which is what your model will predict
train_df["EF_normalized"] = scaler.transform(train_df["EF"].values.reshape(-1, 1))
val_df["EF_normalized"]   = scaler.transform(val_df["EF"].values.reshape(-1, 1))
test_df["EF_normalized"]  = scaler.transform(test_df["EF"].values.reshape(-1, 1))

# --- 5. SAVE THE SCALER ---
# You MUST save this to convert predictions back to real % later
with open(scaler_filename, 'wb') as f:
    pickle.dump(scaler, f)
print(f"Scaler saved to {scaler_filename}")

# --- 6. FINAL VERIFICATION ---
# Check that Training mean is effectively 0 and std is 1
print("\n--- Training Set Stats (Normalized) ---")
print(train_df["EF_normalized"].describe())

# Check that Validation is close to 0/1 but not perfect (natural distribution diff)
print("\n--- Validation Set Stats (Normalized) ---")
print(val_df["EF_normalized"].describe())

# --- 7. CLEANUP ---
# Keep only necessary columns for the dataset loader
cols_to_keep = ["video_path", "EF_normalized", "EF"]
train_final = train_df[cols_to_keep]
val_final   = val_df[cols_to_keep]
test_final  = test_df[cols_to_keep]

print("\nReady for DataLoader. Example row:")
print(train_final.head(1))

Loading CSV...
Data Split: Train (7465), Val (1288), Test (1277)

Scaler Fitted on Train Set.
Mean EF: 55.7776
Std  EF: 12.4064
Scaler saved to ef_scaler.pkl

--- Training Set Stats (Normalized) ---
count    7.465000e+03
mean     8.994814e-17
std      1.000067e+00
min     -3.939140e+00
25%     -3.317867e-01
50%      2.821048e-01
75%      6.620890e-01
max      3.320039e+00
Name: EF_normalized, dtype: float64

--- Validation Set Stats (Normalized) ---
count    1288.000000
mean        0.003777
std         0.992220
min        -3.731570
25%        -0.341866
50%         0.286242
75%         0.673973
max         2.066474
Name: EF_normalized, dtype: float64

Ready for DataLoader. Example row:
                                          video_path  EF_normalized         EF
1  /home/sagemaker-user/user-default-efs/vjepa2/d...       0.267955  59.101988


In [3]:
import os

# 1. Setup Output Directory
output_dir = 'csv'
os.makedirs(output_dir, exist_ok=True)

def write_vjepa_csv(df, filename):
    """
    Writes DataFrame to V-JEPA 2 compatible CSV format:
    - Delimiter: Space
    - Columns: [video_path, EF_normalized]
    - No Header
    - No Index
    """
    output_path = os.path.join(output_dir, filename)
    
    # Select the relevant columns from our previous step
    # Column 1: Video Path
    # Column 2: Normalized Target (EF)
    export_df = df[['video_path', 'EF_normalized']].copy()
    
    # Write to CSV
    export_df.to_csv(
        output_path, 
        sep=' ',       # Space delimiter
        header=False,  # No header row
        index=False    # No index numbers
    )
    
    print(f"✅ Saved {len(df)} rows to: {output_path}")
    
    # Verification: Print the first 2 lines to confirm format
    print(f"--- Preview ({filename}) ---")
    with open(output_path, 'r') as f:
        for _ in range(2):
            print(f.readline().strip())
    print("-" * 30 + "\n")

# 2. Write the files
# using the 'train_final', 'val_final', 'test_final' created in the previous step
write_vjepa_csv(train_final, 'echonet_dynamic_train.csv')
write_vjepa_csv(val_final,   'echonet_dynamic_val.csv')
write_vjepa_csv(test_final,  'echonet_dynamic_test.csv')

✅ Saved 7465 rows to: csv/echonet_dynamic_train.csv
--- Preview (echonet_dynamic_train.csv) ---
/home/sagemaker-user/user-default-efs/vjepa2/data/echonet/echonetdynamic-2/EchoNet-Dynamic/Videos/0X1002E8FBACD08477.avi 0.2679549236578896
/home/sagemaker-user/user-default-efs/vjepa2/data/echonet/echonetdynamic-2/EchoNet-Dynamic/Videos/0X1005D03EED19C65B.avi 0.5308693224419362
------------------------------

✅ Saved 1288 rows to: csv/echonet_dynamic_val.csv
--- Preview (echonet_dynamic_val.csv) ---
/home/sagemaker-user/user-default-efs/vjepa2/data/echonet/echonetdynamic-2/EchoNet-Dynamic/Videos/0X100009310A3BD7FC.avi 1.8313804668987212
/home/sagemaker-user/user-default-efs/vjepa2/data/echonet/echonetdynamic-2/EchoNet-Dynamic/Videos/0X10094BA0A028EAC3.avi -2.489844338267334
------------------------------

✅ Saved 1277 rows to: csv/echonet_dynamic_test.csv
--- Preview (echonet_dynamic_test.csv) ---
/home/sagemaker-user/user-default-efs/vjepa2/data/echonet/echonetdynamic-2/EchoNet-Dynamic/Vid

In [4]:
import os

# --- 1. Define S3 Configuration ---
# Based on your previous sync command, the 'Videos' folder is inside 'EchoNet-Dynamic'
s3_root = "s3://sagemaker-hyperpod-lifecycle-495467399120-usw2/vjepa2-artifacts/data/EchoNet-Dynamic/Videos"

def convert_to_s3_paths(df, s3_base_path):
    """
    Takes a dataframe with local paths, strips the directory, 
    and prepends the S3 bucket path.
    """
    df_s3 = df.copy()
    
    # 1. Extract just the filename (e.g., "video.avi") from the local path
    # 2. F-string it with the S3 base path
    df_s3['video_path'] = df_s3['video_path'].apply(
        lambda x: f"{s3_base_path}/{os.path.basename(x)}"
    )
    return df_s3

# --- 2. Create S3 DataFrames ---
print("Converting paths to S3 URIs...")
train_s3 = convert_to_s3_paths(train_final, s3_root)
val_s3   = convert_to_s3_paths(val_final, s3_root)
test_s3  = convert_to_s3_paths(test_final, s3_root)

# Verify the path looks correct
print(f"Sample S3 Path: {train_s3['video_path'].iloc[0]}")

# --- 3. Save New CSVs ---
# We reuse the write_vjepa_csv function you defined earlier
write_vjepa_csv(train_s3, 'echonet_dynamic_train_s3.csv')
write_vjepa_csv(val_s3,   'echonet_dynamic_val_s3.csv')
write_vjepa_csv(test_s3,  'echonet_dynamic_test_s3.csv')

Converting paths to S3 URIs...
Sample S3 Path: s3://sagemaker-hyperpod-lifecycle-495467399120-usw2/vjepa2-artifacts/data/EchoNet-Dynamic/Videos/0X1002E8FBACD08477.avi
✅ Saved 7465 rows to: csv/echonet_dynamic_train_s3.csv
--- Preview (echonet_dynamic_train_s3.csv) ---
s3://sagemaker-hyperpod-lifecycle-495467399120-usw2/vjepa2-artifacts/data/EchoNet-Dynamic/Videos/0X1002E8FBACD08477.avi 0.2679549236578896
s3://sagemaker-hyperpod-lifecycle-495467399120-usw2/vjepa2-artifacts/data/EchoNet-Dynamic/Videos/0X1005D03EED19C65B.avi 0.5308693224419362
------------------------------

✅ Saved 1288 rows to: csv/echonet_dynamic_val_s3.csv
--- Preview (echonet_dynamic_val_s3.csv) ---
s3://sagemaker-hyperpod-lifecycle-495467399120-usw2/vjepa2-artifacts/data/EchoNet-Dynamic/Videos/0X100009310A3BD7FC.avi 1.8313804668987212
s3://sagemaker-hyperpod-lifecycle-495467399120-usw2/vjepa2-artifacts/data/EchoNet-Dynamic/Videos/0X10094BA0A028EAC3.avi -2.489844338267334
------------------------------

✅ Saved 1277 

# Noise

In [2]:
"""
Generate test CSVs for depth-attenuated EchoNet-Dynamic videos.

This script creates CSV files compatible with V-JEPA 2 for the depth-attenuated
test sets, using the same normalized EF values as the original dataset.

Usage:
    python create_da_test_csvs.py

Requirements:
    pip install pandas numpy scikit-learn
"""

import pandas as pd
import numpy as np
import pickle
import os

# --- 1. CONFIGURATION ---
csv_path = "/home/sagemaker-user/user-default-efs/vjepa2/data/echonet/echonetdynamic-2/EchoNet-Dynamic/FileList.csv"
scaler_path = "/home/sagemaker-user/user-default-efs/vjepa2/data/ef_scaler.pkl"
output_dir = "/home/sagemaker-user/user-default-efs/vjepa2/data/csv"

# Local paths for depth attenuation directories
da_base_path = "/home/sagemaker-user/user-default-efs/vjepa2/data/echonet/echonetdynamic-2"

# S3 paths for depth attenuation directories
s3_base = "s3://sagemaker-hyperpod-lifecycle-495467399120-usw2/vjepa2-artifacts/data"

# Define the depth attenuation configurations
da_configs = [
    ("echonet-dynamic-da075", "da075"),
    ("echonet-dynamic-da150", "da150"),
    ("echonet-dynamic-da215", "da215"),
]

# --- 2. LOAD DATA AND SCALER ---
print("Loading CSV...")
df = pd.read_csv(csv_path)

print("Loading scaler...")
with open(scaler_path, 'rb') as f:
    scaler = pickle.load(f)

print(f"Scaler Mean EF: {scaler.mean_[0]:.4f}")
print(f"Scaler Std EF: {np.sqrt(scaler.var_[0]):.4f}")

Loading CSV...
Loading scaler...
Scaler Mean EF: 55.7776
Scaler Std EF: 12.4064


/opt/conda/lib/python3.12/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.7.2 when using version 1.5.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [3]:
# --- 3. FILTER TO TEST SPLIT ---
test_df = df[df["Split"] == "TEST"].copy()
print(f"\nTest set size: {len(test_df)} videos")

# Normalize EF values using the pre-fitted scaler
test_df["EF_normalized"] = scaler.transform(test_df["EF"].values.reshape(-1, 1))

# --- 4. HELPER FUNCTION ---
os.makedirs(output_dir, exist_ok=True)

def write_vjepa_csv(df, filename):
    """
    Writes DataFrame to V-JEPA 2 compatible CSV format:
    - Delimiter: Space
    - Columns: [video_path, EF_normalized]
    - No Header
    - No Index
    """
    output_path = os.path.join(output_dir, filename)
    
    export_df = df[['video_path', 'EF_normalized']].copy()
    
    export_df.to_csv(
        output_path, 
        sep=' ',
        header=False,
        index=False
    )
    
    print(f"✅ Saved {len(df)} rows to: {output_path}")
    
    # Preview first 2 lines
    print(f"--- Preview ({filename}) ---")
    with open(output_path, 'r') as f:
        for _ in range(2):
            print(f.readline().strip())
    print("-" * 50 + "\n")


Test set size: 1277 videos


In [4]:
# --- 5. CREATE LOCAL PATH CSVs ---
print("\n" + "=" * 60)
print("Creating LOCAL path CSVs")
print("=" * 60 + "\n")

for da_dir, da_suffix in da_configs:
    # Create a copy with updated video paths
    da_test_df = test_df.copy()
    
    # Update paths: original AVI -> new MP4 in DA directory
    da_test_df["video_path"] = da_test_df["FileName"].apply(
        lambda x: os.path.join(da_base_path, da_dir, f"{x}.mp4")
    )
    
    # Verify files exist (check first 3)
    print(f"Checking {da_dir}...")
    missing = 0
    for path in da_test_df["video_path"].head(3).values:
        if not os.path.exists(path):
            print(f"  WARNING: File not found: {path}")
            missing += 1
    if missing == 0:
        print(f"  ✓ Sample files verified")
    
    # Write CSV
    write_vjepa_csv(da_test_df, f"echonet_dynamic_test_{da_suffix}.csv")


Creating LOCAL path CSVs

Checking echonet-dynamic-da075...
  ✓ Sample files verified
✅ Saved 1277 rows to: /home/sagemaker-user/user-default-efs/vjepa2/data/csv/echonet_dynamic_test_da075.csv
--- Preview (echonet_dynamic_test_da075.csv) ---
/home/sagemaker-user/user-default-efs/vjepa2/data/echonet/echonetdynamic-2/echonet-dynamic-da075/0X100CF05D141FF143.mp4 0.014036930992954107
/home/sagemaker-user/user-default-efs/vjepa2/data/echonet/echonetdynamic-2/echonet-dynamic-da075/0X1012703CDC1436FE.mp4 -1.1899719485238105
--------------------------------------------------

Checking echonet-dynamic-da150...
  ✓ Sample files verified
✅ Saved 1277 rows to: /home/sagemaker-user/user-default-efs/vjepa2/data/csv/echonet_dynamic_test_da150.csv
--- Preview (echonet_dynamic_test_da150.csv) ---
/home/sagemaker-user/user-default-efs/vjepa2/data/echonet/echonetdynamic-2/echonet-dynamic-da150/0X100CF05D141FF143.mp4 0.014036930992954107
/home/sagemaker-user/user-default-efs/vjepa2/data/echonet/echonetdy

In [5]:
# --- 6. CREATE S3 PATH CSVs ---
print("\n" + "=" * 60)
print("Creating S3 path CSVs")
print("=" * 60 + "\n")

for da_dir, da_suffix in da_configs:
    # Create a copy with S3 paths
    da_test_s3 = test_df.copy()
    
    # S3 path format
    s3_video_path = f"{s3_base}/{da_dir}"
    da_test_s3["video_path"] = da_test_s3["FileName"].apply(
        lambda x: f"{s3_video_path}/{x}.mp4"
    )
    
    # Write CSV
    write_vjepa_csv(da_test_s3, f"echonet_dynamic_test_{da_suffix}_s3.csv")

# --- 7. SUMMARY ---
print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"\nCreated {len(da_configs) * 2} CSV files in {output_dir}:")
print("\nLocal path CSVs:")
for _, da_suffix in da_configs:
    print(f"  - echonet_dynamic_test_{da_suffix}.csv")
print("\nS3 path CSVs:")
for _, da_suffix in da_configs:
    print(f"  - echonet_dynamic_test_{da_suffix}_s3.csv")

print("\n" + "=" * 60)
print("Done!")
print("=" * 60)


Creating S3 path CSVs

✅ Saved 1277 rows to: /home/sagemaker-user/user-default-efs/vjepa2/data/csv/echonet_dynamic_test_da075_s3.csv
--- Preview (echonet_dynamic_test_da075_s3.csv) ---
s3://sagemaker-hyperpod-lifecycle-495467399120-usw2/vjepa2-artifacts/data/echonet-dynamic-da075/0X100CF05D141FF143.mp4 0.014036930992954107
s3://sagemaker-hyperpod-lifecycle-495467399120-usw2/vjepa2-artifacts/data/echonet-dynamic-da075/0X1012703CDC1436FE.mp4 -1.1899719485238105
--------------------------------------------------

✅ Saved 1277 rows to: /home/sagemaker-user/user-default-efs/vjepa2/data/csv/echonet_dynamic_test_da150_s3.csv
--- Preview (echonet_dynamic_test_da150_s3.csv) ---
s3://sagemaker-hyperpod-lifecycle-495467399120-usw2/vjepa2-artifacts/data/echonet-dynamic-da150/0X100CF05D141FF143.mp4 0.014036930992954107
s3://sagemaker-hyperpod-lifecycle-495467399120-usw2/vjepa2-artifacts/data/echonet-dynamic-da150/0X1012703CDC1436FE.mp4 -1.1899719485238105
------------------------------------------

# Gaussian Shadow

In [5]:
"""
Generate test CSVs for depth-attenuated EchoNet-Dynamic videos.

This script creates CSV files compatible with V-JEPA 2 for the depth-attenuated
test sets, using the same normalized EF values as the original dataset.

Usage:
    python create_da_test_csvs.py

Requirements:
    pip install pandas numpy scikit-learn
"""

import pandas as pd
import numpy as np
import pickle
import os

# --- 1. CONFIGURATION ---
csv_path = "/home/sagemaker-user/user-default-efs/vjepa2/data/echonet/echonetdynamic-2/EchoNet-Dynamic/FileList.csv"
scaler_path = "/home/sagemaker-user/user-default-efs/vjepa2/data/ef_scaler.pkl"
output_dir = "/home/sagemaker-user/user-default-efs/vjepa2/data/csv"

# Local paths for depth attenuation directories
da_base_path = "/home/sagemaker-user/user-default-efs/vjepa2/data/echonet/echonetdynamic-2"

# S3 paths for depth attenuation directories
s3_base = "s3://sagemaker-hyperpod-lifecycle-495467399120-usw2/vjepa2-artifacts/data"

# Define the depth attenuation configurations
da_configs = [
    ("echonet-dynamic-gs-low", "gs-low"),
    ("echonet-dynamic-gs-med", "gs-med"),
    ("echonet-dynamic-gs-high", "gs-high"),
]

# --- 2. LOAD DATA AND SCALER ---
print("Loading CSV...")
df = pd.read_csv(csv_path)

print("Loading scaler...")
with open(scaler_path, 'rb') as f:
    scaler = pickle.load(f)

print(f"Scaler Mean EF: {scaler.mean_[0]:.4f}")
print(f"Scaler Std EF: {np.sqrt(scaler.var_[0]):.4f}")

Loading CSV...
Loading scaler...
Scaler Mean EF: 55.7776
Scaler Std EF: 12.4064


/opt/conda/lib/python3.12/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.7.2 when using version 1.7.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [6]:
# --- 3. FILTER TO TEST SPLIT ---
test_df = df[df["Split"] == "TEST"].copy()
print(f"\nTest set size: {len(test_df)} videos")

# Normalize EF values using the pre-fitted scaler
test_df["EF_normalized"] = scaler.transform(test_df["EF"].values.reshape(-1, 1))

# --- 4. HELPER FUNCTION ---
os.makedirs(output_dir, exist_ok=True)

def write_vjepa_csv(df, filename):
    """
    Writes DataFrame to V-JEPA 2 compatible CSV format:
    - Delimiter: Space
    - Columns: [video_path, EF_normalized]
    - No Header
    - No Index
    """
    output_path = os.path.join(output_dir, filename)
    
    export_df = df[['video_path', 'EF_normalized']].copy()
    
    export_df.to_csv(
        output_path, 
        sep=' ',
        header=False,
        index=False
    )
    
    print(f"✅ Saved {len(df)} rows to: {output_path}")
    
    # Preview first 2 lines
    print(f"--- Preview ({filename}) ---")
    with open(output_path, 'r') as f:
        for _ in range(2):
            print(f.readline().strip())
    print("-" * 50 + "\n")


Test set size: 1277 videos


In [7]:
# --- 5. CREATE LOCAL PATH CSVs ---
print("\n" + "=" * 60)
print("Creating LOCAL path CSVs")
print("=" * 60 + "\n")

for da_dir, da_suffix in da_configs:
    # Create a copy with updated video paths
    da_test_df = test_df.copy()
    
    # Update paths: original AVI -> new MP4 in DA directory
    da_test_df["video_path"] = da_test_df["FileName"].apply(
        lambda x: os.path.join(da_base_path, da_dir, f"{x}.mp4")
    )
    
    # Verify files exist (check first 3)
    print(f"Checking {da_dir}...")
    missing = 0
    for path in da_test_df["video_path"].head(3).values:
        if not os.path.exists(path):
            print(f"  WARNING: File not found: {path}")
            missing += 1
    if missing == 0:
        print(f"  ✓ Sample files verified")
    
    # Write CSV
    write_vjepa_csv(da_test_df, f"echonet_dynamic_test_{da_suffix}.csv")


Creating LOCAL path CSVs

Checking echonet-dynamic-gs-low...
  ✓ Sample files verified
✅ Saved 1277 rows to: /home/sagemaker-user/user-default-efs/vjepa2/data/csv/echonet_dynamic_test_gs-low.csv
--- Preview (echonet_dynamic_test_gs-low.csv) ---
/home/sagemaker-user/user-default-efs/vjepa2/data/echonet/echonetdynamic-2/echonet-dynamic-gs-low/0X100CF05D141FF143.mp4 0.014036930992954107
/home/sagemaker-user/user-default-efs/vjepa2/data/echonet/echonetdynamic-2/echonet-dynamic-gs-low/0X1012703CDC1436FE.mp4 -1.1899719485238105
--------------------------------------------------

Checking echonet-dynamic-gs-med...
  ✓ Sample files verified
✅ Saved 1277 rows to: /home/sagemaker-user/user-default-efs/vjepa2/data/csv/echonet_dynamic_test_gs-med.csv
--- Preview (echonet_dynamic_test_gs-med.csv) ---
/home/sagemaker-user/user-default-efs/vjepa2/data/echonet/echonetdynamic-2/echonet-dynamic-gs-med/0X100CF05D141FF143.mp4 0.014036930992954107
/home/sagemaker-user/user-default-efs/vjepa2/data/echonet/

In [4]:
# --- 6. CREATE S3 PATH CSVs ---
print("\n" + "=" * 60)
print("Creating S3 path CSVs")
print("=" * 60 + "\n")

for da_dir, da_suffix in da_configs:
    # Create a copy with S3 paths
    da_test_s3 = test_df.copy()
    
    # S3 path format
    s3_video_path = f"{s3_base}/{da_dir}"
    da_test_s3["video_path"] = da_test_s3["FileName"].apply(
        lambda x: f"{s3_video_path}/{x}.mp4"
    )
    
    # Write CSV
    write_vjepa_csv(da_test_s3, f"echonet_dynamic_test_{da_suffix}_s3.csv")

# --- 7. SUMMARY ---
print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"\nCreated {len(da_configs) * 2} CSV files in {output_dir}:")
print("\nLocal path CSVs:")
for _, da_suffix in da_configs:
    print(f"  - echonet_dynamic_test_{da_suffix}.csv")
print("\nS3 path CSVs:")
for _, da_suffix in da_configs:
    print(f"  - echonet_dynamic_test_{da_suffix}_s3.csv")

print("\n" + "=" * 60)
print("Done!")
print("=" * 60)


Creating S3 path CSVs

✅ Saved 1277 rows to: /home/sagemaker-user/user-default-efs/vjepa2/data/csv/echonet_dynamic_test_gs-low_s3.csv
--- Preview (echonet_dynamic_test_gs-low_s3.csv) ---
s3://sagemaker-hyperpod-lifecycle-495467399120-usw2/vjepa2-artifacts/data/echonet-dynamic-gs-low/0X100CF05D141FF143.mp4 0.014036930992954107
s3://sagemaker-hyperpod-lifecycle-495467399120-usw2/vjepa2-artifacts/data/echonet-dynamic-gs-low/0X1012703CDC1436FE.mp4 -1.1899719485238105
--------------------------------------------------

✅ Saved 1277 rows to: /home/sagemaker-user/user-default-efs/vjepa2/data/csv/echonet_dynamic_test_gs-high_s3.csv
--- Preview (echonet_dynamic_test_gs-high_s3.csv) ---
s3://sagemaker-hyperpod-lifecycle-495467399120-usw2/vjepa2-artifacts/data/echonet-dynamic-gs-med/0X100CF05D141FF143.mp4 0.014036930992954107
s3://sagemaker-hyperpod-lifecycle-495467399120-usw2/vjepa2-artifacts/data/echonet-dynamic-gs-med/0X1012703CDC1436FE.mp4 -1.1899719485238105
--------------------------------

# Pediatric

In [7]:
import pandas as pd
import os

# --- 1. CONFIGURATION ---
base_dir = "/home/sagemaker-user/user-default-efs/vjepa2/data/echonetpediatric/pediatric_echo_avi/pediatric_echo_avi/A4C"
csv_path = os.path.join(base_dir, "FileList.csv")

# --- 2. LOAD DATA ---
print(f"Loading CSV from: {csv_path}")
df = pd.read_csv(csv_path)

# --- 3. DEBUGGING PRINTS ---
print("\n--- Columns Found ---")
print(df.columns.tolist())

print("\n--- First 5 Rows ---")
print(df.head())

if "Split" in df.columns:
    print("\n--- Unique Values in 'Split' Column ---")
    print(df["Split"].unique())
else:
    print("\n⚠️ 'Split' column is MISSING. We will need to create random splits.")

print(f"\nTotal rows in CSV: {len(df)}")

Loading CSV from: /home/sagemaker-user/user-default-efs/vjepa2/data/echonetpediatric/pediatric_echo_avi/pediatric_echo_avi/A4C/FileList.csv

--- Columns Found ---
['FileName', 'EF', 'Sex', 'Age', 'Weight', 'Height', 'Split']

--- First 5 Rows ---
                         FileName     EF Sex  Age  Weight  Height  Split
0  CR32a7555-CR32a7582-000039.avi  40.83   F    0    10.2    68.5      5
1  CR32a7555-CR32a97af-000033.avi  52.62   F    1    15.5    85.0      5
2  CR32a7555-CR32a97e1-000024.avi  24.85   F    0     4.0    56.0      5
3  CR32a7555-CR32a9850-000040.avi  50.96   F    4    18.0    99.0      5
4  CR32a7555-CR32a988d-000034.avi  56.76   F    0    13.2    75.0      5

--- Unique Values in 'Split' Column ---
[5 7 1 9 0 2 4 8 3 6]

Total rows in CSV: 3284


In [8]:
import os

# --- CONFIG ---
# Ensure this matches your previous base_dir
base_dir = "/home/sagemaker-user/user-default-efs/vjepa2/data/echonetpediatric/pediatric_echo_avi/pediatric_echo_avi/A4C"
video_root = os.path.join(base_dir, "Videos")

# --- 1. CONSTRUCT PATHS ---
# FIX: The FileName column already has .avi, so we just join the path
df["video_path"] = df["FileName"].apply(lambda x: os.path.join(video_root, x))

# Verify the first path looks correct now
print(f"Sample Video Path: {df['video_path'].iloc[0]}")

# --- 2. CREATE SPLITS ---
# We map the 10 folds (0-9) to standard sets:
# Folds 0-7 (80%) -> TRAIN
# Fold  8   (10%) -> VAL
# Fold  9   (10%) -> TEST

train_folds = [0, 1, 2, 3, 4, 5, 6, 7]
val_folds   = [8]
test_folds  = [9]

train_df = df[df["Split"].isin(train_folds)].copy()
val_df   = df[df["Split"].isin(val_folds)].copy()
test_df  = df[df["Split"].isin(test_folds)].copy()

print(f"\nFINAL SPLIT COUNTS:")
print(f"Train: {len(train_df)} (Folds {train_folds})")
print(f"Val:   {len(val_df)}   (Fold {val_folds})")
print(f"Test:  {len(test_df)}  (Fold {test_folds})")

# Stop if something went wrong
if len(train_df) == 0:
    raise ValueError("Train set is empty! Check the fold logic.")

Sample Video Path: /home/sagemaker-user/user-default-efs/vjepa2/data/echonetpediatric/pediatric_echo_avi/pediatric_echo_avi/A4C/Videos/CR32a7555-CR32a7582-000039.avi

FINAL SPLIT COUNTS:
Train: 2580 (Folds [0, 1, 2, 3, 4, 5, 6, 7])
Val:   336   (Fold [8])
Test:  368  (Fold [9])


In [9]:
import numpy as np
import pickle
from sklearn.preprocessing import StandardScaler

# --- CONFIG ---
scaler_filename = "pediatric_ef_scaler.pkl"
output_dir = 'csv_pediatric'
s3_root = "s3://sagemaker-hyperpod-lifecycle-495467399120-usw2/vjepa2-artifacts/data/echonetpediatric/pediatric_echo_avi/pediatric_echo_avi/A4C/Videos"

os.makedirs(output_dir, exist_ok=True)

# --- 1. NORMALIZE (EF) ---
scaler = StandardScaler()

# Fit on TRAIN only
train_ef_values = train_df["EF"].values.reshape(-1, 1)
scaler.fit(train_ef_values)

print(f"\nScaler Fitted.")
print(f"Mean EF: {scaler.mean_[0]:.4f}")
print(f"Std  EF: {np.sqrt(scaler.var_[0]):.4f}")

# Transform ALL sets
train_df["EF_normalized"] = scaler.transform(train_df["EF"].values.reshape(-1, 1))
val_df["EF_normalized"]   = scaler.transform(val_df["EF"].values.reshape(-1, 1))
test_df["EF_normalized"]  = scaler.transform(test_df["EF"].values.reshape(-1, 1))

# Save Scaler
with open(scaler_filename, 'wb') as f:
    pickle.dump(scaler, f)
print(f"✅ Scaler saved to {scaler_filename}")

# --- 2. FUNCTION TO WRITE CSV ---
def write_vjepa_csv(df_in, filename):
    path = os.path.join(output_dir, filename)
    # Write: [video_path] [target]
    df_in[['video_path', 'EF_normalized']].to_csv(path, sep=' ', header=False, index=False)
    print(f"Saved: {path}")

# --- 3. SAVE LOCAL CSVs ---
print("\n--- Saving Local CSVs ---")
write_vjepa_csv(train_df, 'echonet_pediatric_train.csv')
write_vjepa_csv(val_df,   'echonet_pediatric_val.csv')
write_vjepa_csv(test_df,  'echonet_pediatric_test.csv')

# --- 4. GENERATE S3 CSVs ---
print("\n--- Saving S3 CSVs ---")
def convert_to_s3_paths(df_in, s3_base):
    df_s3 = df_in.copy()
    # Replace local path with S3 bucket path
    df_s3['video_path'] = df_s3['video_path'].apply(
        lambda x: f"{s3_base}/{os.path.basename(x)}"
    )
    return df_s3

train_s3 = convert_to_s3_paths(train_df, s3_root)
val_s3   = convert_to_s3_paths(val_df, s3_root)
test_s3  = convert_to_s3_paths(test_df, s3_root)

write_vjepa_csv(train_s3, 'echonet_pediatric_train_s3.csv')
write_vjepa_csv(val_s3,   'echonet_pediatric_val_s3.csv')
write_vjepa_csv(test_s3,  'echonet_pediatric_test_s3.csv')


Scaler Fitted.
Mean EF: 61.0272
Std  EF: 10.4406
✅ Scaler saved to pediatric_ef_scaler.pkl

--- Saving Local CSVs ---
Saved: csv_pediatric/echonet_pediatric_train.csv
Saved: csv_pediatric/echonet_pediatric_val.csv
Saved: csv_pediatric/echonet_pediatric_test.csv

--- Saving S3 CSVs ---
Saved: csv_pediatric/echonet_pediatric_train_s3.csv
Saved: csv_pediatric/echonet_pediatric_val_s3.csv
Saved: csv_pediatric/echonet_pediatric_test_s3.csv


In [2]:
import os
import glob
import pandas as pd
import xml.etree.ElementTree as ET
from tqdm.notebook import tqdm

# ---------------- CONFIG ----------------
SOURCE_ROOT = os.path.join(".", "unmasked")
REFERENCE_CSV = "labels_patient_split.csv" # The source of truth for splits

# ---------------- HELPERS ----------------
def map_quality(raw_val):
    if not raw_val: return None
    v = str(raw_val).lower()
    
    # --- BINARY MAPPING STRATEGY ---
    # Class 0 (Fail): Unusable/Poor. (Old "Low" + Scores 0, 1, 2)
    if 'low' in v or any(s in v for s in ['0', '1', '2']):
        return "discard"
        
    # Class 1 (Pass): Diagnostic/Good. (Old "Med" + "High" + Scores 3, 4, 5, 6)
    # Merging these eliminates the noise that caused your model collapse.
    if 'medium' in v or 'high' in v or any(s in v for s in ['3', '4', '5', '6']):
        return "keep"
        
    return None

def parse_all_tasks(xml_path, job_id, job_folder_path):
    if not os.path.exists(xml_path): return []
    tree = ET.parse(xml_path)
    root = tree.getroot()
    data = []

    for image in root.findall('image'):
        rel_path = image.get('name')
        base, ext = os.path.splitext(rel_path)
        masked_path = os.path.join(job_folder_path, f"{base}_masked{ext}")
        
        tag = image.find("tag[@label='TTE']")
        if tag is None: continue

        attrs = {attr.get('name'): attr.text for attr in tag.findall('attribute')}
        data.append({
            "filename": masked_path,
            "job_id": int(job_id), # Ensure job_id is int for matching
            "color": attrs.get('Color'),
            "zoom": attrs.get('Zoom'),
            "quality": map_quality(attrs.get('Quality'))
        })
    return data

# 1. Load the reference split mapping
print(f"Loading reference splits from {REFERENCE_CSV}...")
ref_df = pd.read_csv(REFERENCE_CSV)

# Ensure we have a clean mapping of job_id -> split
# (Handling the potential duplicate 'split' columns you mentioned earlier)
if 'split' not in ref_df.columns:
    split_col = [c for c in ref_df.columns if c.startswith('split')][0]
    ref_df = ref_df.rename(columns={split_col: 'split'})

split_map = ref_df[['job_id', 'split']].drop_duplicates().set_index('job_id')['split'].to_dict()

# 2. Gather all raw attribute data
all_data = []
job_folders = sorted(glob.glob(os.path.join(SOURCE_ROOT, "job_*")))
for folder in tqdm(job_folders, desc="Parsing XMLs"):
    jid = folder.split("_")[-1]
    all_data.extend(parse_all_tasks(os.path.join(folder, "annotations.xml"), jid, folder))

df_master = pd.DataFrame(all_data)

# 3. Apply the reference split to the master dataframe
df_master['split'] = df_master['job_id'].map(split_map)

# 4. Create individual CSVs
tasks = {
    "color": "labels_color.csv",
    "zoom": "labels_zoom.csv",
    "quality": "labels_quality.csv"
}

print("\n--- Generation Summary ---")
for task_key, filename in tasks.items():
    # Filter for valid rows, drop rows with no split mapping (if any), and format
    subset = df_master[df_master[task_key].notna() & df_master['split'].notna()].copy()
    subset = subset.rename(columns={task_key: "label"})
    subset = subset[["filename", "label", "job_id", "split"]]
    
    subset.to_csv(filename, index=False)
    
    print(f"\nSaved {filename} ({len(subset)} rows)")
    print(pd.crosstab(subset["label"], subset["split"]))

Loading reference splits from labels_patient_split.csv...


Parsing XMLs:   0%|          | 0/70 [00:00<?, ?it/s]


--- Generation Summary ---

Saved labels_color.csv (23116 rows)
split  test  train   val
label                   
No     4785   4576  5084
Yes    2731   2820  3120

Saved labels_zoom.csv (23116 rows)
split  test  train   val
label                   
Full   6136   5920  6376
Large  1213   1322  1619
Small   167    154   209

Saved labels_quality.csv (23111 rows)
split    test  train   val
label                     
discard  2413   2236  2442
keep     5102   5159  5759
